# HRAI API demo notebook

In [ ]:
# from main import QueryRequest
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [ ]:
import os, json, random, textwrap, requests
from pathlib import Path

from config import conf
from load import get_encoder
from query import query_type, extract_from_resume
from suggestions import get_expanded_skills, get_domain_reports
from pos_extraction import text_to_ngrams

from dotenv import load_dotenv
import faiss

BASE_URL = 'http://127.0.0.1:8000'

## konfigurace modelu a metadat pro lokální volání
\*pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env

In [ ]:
load_dotenv()
db = faiss.read_index(os.path.join(conf.db_dir, f"all.index"))
with open(os.path.join(conf.data_dir, f"key_to_ent.json"),'r') as f:
    metadata = json.loads(f.read())
model = get_encoder()

In [ ]:
def print_skill(s):
    print(f"{s['label']}")
    if s['description']: print(f"informace: {s['description']}\n")


def print_occupation(occ):
    print(
        f"{occ['label']}\n"
        f"skóre: {round(occ['cosine_score'],5)}"
    )
    if occ['description']: print(f"info: {occ['description']}\n")


def print_missing_skills(skills):
    for s in skills:
        if 'missing_skills' in s and len(s['missing_skills']) == 0: continue
        if 'occupation' in s: print_occupation(s['occupation'])
        if 'missing_skills' not in s: continue
        essential = [s for s in s['missing_skills']if s['relation_type'] == 'essential']
        optional = [s for s in s['missing_skills']if s['relation_type'] == 'optional']

        print('Základní dovednosti/znalosti pro výkon pozice:')
        for e in essential: print_skill(e)

        print('\nDalší užitečné dovednosti:')
        for o in optional: print_skill(o)
        print('-'*100)


def print_domain_info(d):
    for occ in d['occupations']:
        print_occupation(occ['occupation'])
        if 'missing_skills' in occ and len(occ['missing_skills']) > 0:
            print('')
            print_missing_skills([occ])
    print('-'*100+'\n')

## převedení textu na ngramy

In [ ]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
print(textwrap.shorten(text, width=2000))

In [ ]:
ngrams = text_to_ngrams(text)
# print('\n'.join(ngrams[:20])+'\n[...]')
print(f'ngramy: {len(ngrams)}\n'
      f'text: {len(text.split())}'
      )
print('\n'.join(ngrams))

## API endpointy: /resume/skills a /resume/domains (pouze PDF)

In [ ]:
def get_skills(filename: str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/skills", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )
    print_missing_skills(res.json())


def get_domains(filename:str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/domains", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )

    for d in res.json():
        print(f"oblast: {d['domain']}\n"
          f"počet zaměstnání se shodou: {d['occ_count']}\n"
          )
        if 'occupations' not in d: continue
        if len(d['occupations']) <1: continue
        print_domain_info(d)


In [ ]:
get_skills('resume.pdf')

In [ ]:
get_domains('resume.docx')

## API endpointy: /text/skills a /text/domains

In [ ]:
text_req = {
    'occupations': ['software developer', 'data analyst'],
    'skills': ['python', 'strojové učení', 'sql databáze']
}
skills_resp = requests.post(f"{BASE_URL}/text/skills", json=text_req)
print(f"endpoint: /text/skills | status: {skills_resp.status_code}\n")
print_missing_skills(skills_resp.json())


domains_resp = requests.post(f"{BASE_URL}/text/domains", json=text_req)
print(f"endpoint: /text/domains | status: {domains_resp.status_code}\n")
for d in domains_resp.json():
    print(f"oblast: {d['domain']}\n"
          f"počet zaměstnání se shodou: {d['occ_count']}\n"
          )
    if 'occupations' not in d or len(d['occupations']) < 1: continue
    print_domain_info(d)

## API endpoint: /query (každý typ dotazu)

In [ ]:
from main import QueryRequest
# TODO fix example request

query = QueryRequest(
    query='manažerka v mcdonalds',
    query_type='occupation',
    min_set_score=0.7
)

resp = requests.post(f"{BASE_URL}/query", json=query.model_dump())
print({'endpoint': '/query', 'status': resp.status_code, 'json': resp.json()})

## Přímé volání query_type pro všechny ent_type

In [ ]:
entry = ['rybář']
ent_type = 'occupation'

results = query_type(db=db, metadata=metadata, model=model, ents=entry, ent_type=ent_type, min_score=0.5)
print(f"{entry} ({ent_type})")
print(results)

## Pipeline: extract_from_resume -> get_expanded_skills -> get_domain_reports

In [ ]:
random_resume = random.choice(resumes)
print(textwrap.shorten(f'sample text: {random_resume}', width=2000, placeholder="..."))
print('-'*80)

extracted_entities = extract_from_resume(db, metadata, model, random_resume)
print(textwrap.shorten(f'extracted entities: {extracted_entities}', width=2000, placeholder="..."))
print('-'*80)

skill_suggestions = get_expanded_skills(metadata, extracted_entities)
print(textwrap.shorten(f'skill suggestions: {skill_suggestions}', width=2000, placeholder="..."))

domain_reports = get_domain_reports(skill_suggestions)
print(textwrap.shorten(f'domain stats: {domain_reports}', width=2000, placeholder="..."))